# Multifamily Supply Analysis

This notebook analyzes multifamily inventory and competitive supply in the Clarkston trade area.

**Workflow:** load source data, clean fields, derive KPIs, and summarize the local supply picture.


In [1]:
import pandas as pd

# Read in xlsx file
file_path = "../data/Residential/supply/MultifamilyDataGrid_5mile_ALLBR.xlsx"
df = pd.read_excel(file_path)

# Preview the first few rows
df.head()


,Period,Inventory Bldgs,Inventory Units,Inventory Avg SF,Asking Rent Per Unit,Asking Rent Per SF,Asking Rent % Growth/Yr,Effective Rent Per Unit,Effective Rent Per SF,Effective Rent % Growth/Yr,...,Occupancy Percent,Occupancy % Growth/Yr,Absorption Units,Absorption Percent,Under Construction Bldgs,Under Construction Units,Under Construction Percent,Deliveries Bldgs,Deliveries Units,Deliveries Percent
0,2026 YTD,39,9834,948,1858,1.96,0.002,1806,1.90,0.006,...,0.907,0.031,27,0.003,1,372,0.038,-,-,0.000
1,2025,39,9834,948,1866,1.97,0.012,1820,1.92,0.004,...,0.904,0.043,417,0.042,1,372,0.038,1,60,0.006
2,2024,38,9774,948,1845,1.94,-0.001,1812,1.91,-0.003,...,0.862,-0.011,750,0.077,2,432,0.044,3,979,0.100
3,2023,35,8795,946,1846,1.95,-0.038,1818,1.92,-0.041,...,0.872,-0.009,491,0.056,4,1039,0.118,3,649,0.074
4,2022,32,8146,938,1918,2.02,0.004,1896,2.00,-0.004,...,0.881,-0.074,-97,-0.012,6,1628,0.2,2,530,0.065


In [2]:
#print columns of df
print(df.columns)

Index(['  Period', 'Inventory Bldgs', 'Inventory Units', 'Inventory Avg SF',
       'Asking Rent Per Unit', 'Asking Rent Per SF', 'Asking Rent % Growth/Yr',
       'Effective Rent Per Unit', 'Effective Rent Per SF',
       'Effective Rent % Growth/Yr', 'Effective Rent Concessions %',
       'Vacancy Units', 'Vacancy Percent', 'Vacancy % Growth/Yr',
       'Occupancy Units', 'Occupancy Percent', 'Occupancy % Growth/Yr',
       'Absorption Units', 'Absorption Percent', 'Under Construction Bldgs',
       'Under Construction Units', 'Under Construction Percent',
       'Deliveries Bldgs', 'Deliveries Units', 'Deliveries Percent'],
      dtype='object')


In [3]:
# Remove spaces only from the first column name that contains 'Period'
period_col_idx = next(i for i, col in enumerate(df.columns) if "Period" in col)
df.columns.values[period_col_idx] = df.columns[period_col_idx].replace(' ', '')


In [4]:
import pandas as pd

# 1. Data Cleaning: Convert strings and '-' to numeric values
cols = ['Inventory Units', 'Deliveries Units', 'Under Construction Units']
for col in cols:
    df[col] = pd.to_numeric(df[col].astype(str).replace('-', '0'), errors='coerce').fillna(0)

# 2. Verify Inventory Math (Year-over-Year change should equal Deliveries)
# Since the data is sorted by Period descending, we shift(-1) to get the previous year
df['Calculated_Deliveries'] = df['Inventory Units'] - df['Inventory Units'].shift(-1)
df['Inventory_Check'] = df['Calculated_Deliveries'] == df['Deliveries Units']

# 3. Calculate 'Construction Starts' 
# This shows how many NEW units started construction in that year
df['New_Starts'] = df['Under Construction Units'] - (df['Under Construction Units'].shift(-1) - df['Deliveries Units'])

# 4. Supply Projection for 2029
# Total Potential Inventory = Current Inventory + Current Under Construction + Future Planned Starts
current_inventory = df.loc[0, 'Inventory Units']
active_pipeline = df.loc[0, 'Under Construction Units']
projected_supply_2029 = current_inventory + active_pipeline

print(f"Current Standing Inventory: {current_inventory}")
print(f"Total Known Pipeline (Deliveries 2026-2028): {active_pipeline}")
print(f"Minimum Expected 2029 Inventory (excluding new starts): {projected_supply_2029}")
print("\nMath Validation Table:")
print(df[['Period', 'Inventory Units', 'Deliveries Units', 'Inventory_Check', 'New_Starts']].head())

Current Standing Inventory: 9834
Total Known Pipeline (Deliveries 2026-2028): 372
Minimum Expected 2029 Inventory (excluding new starts): 10206

Math Validation Table:
     Period  Inventory Units  Deliveries Units  Inventory_Check  New_Starts
0  2026 YTD             9834                 0             True         0.0
1      2025             9834                60             True         0.0
2      2024             9774               979             True       372.0
3      2023             8795               649             True        60.0
4      2022             8146               530             True      1434.0


In [5]:
import pandas as pd

# 1. Data Cleaning: Convert strings and '-' to numeric values
cols = ['Inventory Units', 'Deliveries Units', 'Under Construction Units']
for col in cols:
    df[col] = pd.to_numeric(df[col].astype(str).replace('-', '0'), errors='coerce').fillna(0)

# 2. Verify Inventory Math (Year-over-Year change should equal Deliveries)
# Since the data is sorted by Period descending, we shift(-1) to get the previous year
df['Calculated_Deliveries'] = df['Inventory Units'] - df['Inventory Units'].shift(-1)
df['Inventory_Check'] = df['Calculated_Deliveries'] == df['Deliveries Units']

# 3. Calculate 'Construction Starts' 
# Logic: Starts = Current UC - (Previous UC - Current Deliveries)
df['New_Starts'] = df['Under Construction Units'] - (df['Under Construction Units'].shift(-1) - df['Deliveries Units'])

# Select only the columns necessary for the supply-side audit
df_check = df[['Period', 'Inventory Units', 'Deliveries Units', 'Under Construction Units', 'Inventory_Check', 'New_Starts']].copy()

# Display the focused dataframe
print("df_check: Pipeline Audit Table")
df_check

df_check: Pipeline Audit Table


,Period,Inventory Units,Deliveries Units,Under Construction Units,Inventory_Check,New_Starts
0,2026 YTD,9834,0,372,True,0.0
1,2025,9834,60,372,True,0.0
2,2024,9774,979,432,True,372.0
3,2023,8795,649,1039,True,60.0
4,2022,8146,530,1628,True,1434.0
5,2021,7616,332,724,True,240.0
6,2020,7284,768,816,True,726.0
7,2019,6516,668,858,True,101.0
8,2018,5848,321,1425,True,757.0
9,2017,5527,399,989,True,611.0


In [6]:
# Print the entire column set to wide display
with pd.option_context('display.max_columns', None, 'display.width', 2000):
    print(df.head())

     Period  Inventory Bldgs  Inventory Units  Inventory Avg SF  Asking Rent Per Unit  Asking Rent Per SF Asking Rent % Growth/Yr  Effective Rent Per Unit  Effective Rent Per SF Effective Rent % Growth/Yr  Effective Rent Concessions %  Vacancy Units  Vacancy Percent Vacancy % Growth/Yr  Occupancy Units  Occupancy Percent Occupancy % Growth/Yr  Absorption Units  Absorption Percent Under Construction Bldgs  Under Construction Units Under Construction Percent Deliveries Bldgs  Deliveries Units  Deliveries Percent  Calculated_Deliveries  Inventory_Check  New_Starts
0  2026 YTD               39             9834               948                  1858                1.96                   0.002                     1806                   1.90                      0.006                         0.028            910            0.093              -0.031             8864              0.907                 0.031                27               0.003                        1                       37

In [7]:
# 1. Calculate New Starts (The "Inflow")
df['New_Starts'] = df['Under Construction Units'] - (df['Under Construction Units'].shift(-1) - df['Deliveries Units'])

# 2. Calculate Market Digestion (Supply vs. Demand Balance)
# A positive number means demand exceeded new supply (the market "digested" the units).
df['Net_Supply_Demand'] = df['Absorption Units'] - df['Deliveries Units']

# 3. Create the focused audit table
df_digest = df[['Period', 'Deliveries Units', 'Absorption Units', 'Net_Supply_Demand', 
                'Under Construction Units', 'New_Starts', 'Occupancy Percent']]

# Sorting chronologically to tell the story
print(df_digest.sort_values('Period'))

      Period  Deliveries Units  Absorption Units  Net_Supply_Demand  \
26      2000               250               118               -132   
25      2001                 2                87                 85   
24      2002               537               517                -20   
23      2003                 0               -19                -19   
22      2004               342               290                -52   
21      2005                 0                16                 16   
20      2006               915               652               -263   
19      2007                 0               149                149   
18      2008               268               282                 14   
17      2009               622               503               -119   
16      2010               399               335                -64   
15      2011                 0               105                105   
14      2012                 0               123                123   
13    

In [8]:
# 1. Calculate Average Annual Absorption (5-Year Baseline)
# We exclude YTD data to get a clean 12-month average.
avg_abs_5yr = df[df['Period'] != '2026 YTD'].head(5)['Absorption Units'].mean()
monthly_pace_5yr = avg_abs_5yr / 12

# Now, also calculate monthly demand pace for the entire available history (excluding 2026 YTD)
avg_abs_all = df[df['Period'] != '2026 YTD']['Absorption Units'].mean()
monthly_pace_all = avg_abs_all / 12

# 2. Same analysis for Asking Rent % Growth/Yr
df['Asking Rent % Growth/Yr'] = pd.to_numeric(df['Asking Rent % Growth/Yr'], errors='coerce')

# 5-Year Average (excluding YTD, just like above)
avg_rent_growth_5yr = df[df['Period'] != '2026 YTD'].head(5)['Asking Rent % Growth/Yr'].mean()
# All available years average (excluding YTD)
avg_rent_growth_all = df[df['Period'] != '2026 YTD']['Asking Rent % Growth/Yr'].mean()
# Last 2 years (excluding YTD)
avg_rent_growth_2yr = df[df['Period'] != '2026 YTD'].head(2)['Asking Rent % Growth/Yr'].mean()

# 3. Total Competition Pile
total_comp = df.loc[0, 'Vacancy Units'] + df.loc[0, 'Under Construction Units']

# 4. Months of Supply
# How long to clear the "Pile" at current 5-yr average demand levels?
mos_5yr = total_comp / monthly_pace_5yr
# How long to clear the "Pile" at all-history average demand levels?
mos_all = total_comp / monthly_pace_all

print(f"Monthly Market Demand Pace (5-yr baseline): {monthly_pace_5yr:.1f} units/month")
print(f"Monthly Market Demand Pace (all history): {monthly_pace_all:.1f} units/month")
print(f"Months of Supply Remaining (5-yr pace): {mos_5yr:.1f} months")
print(f"Months of Supply Remaining (all history pace): {mos_all:.1f} months")
print(f"Asking Rent % Growth/Yr, 5-yr average: {avg_rent_growth_5yr:.2%}")
print(f"Asking Rent % Growth/Yr, all history average: {avg_rent_growth_all:.2%}")
print(f"Asking Rent % Growth/Yr, last 2 years: {avg_rent_growth_2yr:.2%}")

Monthly Market Demand Pace (5-yr baseline): 43.4 units/month
Monthly Market Demand Pace (all history): 27.2 units/month
Months of Supply Remaining (5-yr pace): 29.5 months
Months of Supply Remaining (all history pace): 47.1 months
Asking Rent % Growth/Yr, 5-yr average: 2.48%
Asking Rent % Growth/Yr, all history average: 1.00%
Asking Rent % Growth/Yr, last 2 years: 0.55%


In [9]:
# calculcate the average monthly absorption 

In [10]:
import pandas as pd

# 1. Clean the Vacancy Percent column
df['Vacancy Percent'] = pd.to_numeric(df['Vacancy Percent'], errors='coerce')

# 2. Historical Averages
# Full Cycle (2000-2025) - Excluding YTD
avg_vac_full = df[df['Period'] != '2026 YTD']['Vacancy Percent'].mean()

# Last 10 Full Years (2016-2025)
avg_vac_10yr = df[df['Period'] != '2026 YTD'].head(10)['Vacancy Percent'].mean()

# "Stabilized Era" (2014-2021)
# This excludes the recent 2022-2024 supply shock to see the true market ceiling.
avg_vac_stabilized = df[(df['Period'].astype(str) >= '2014') & (df['Period'].astype(str) <= '2021')]['Vacancy Percent'].mean()

print(f"Full History Average Vacancy (26 yrs): {avg_vac_full:.1%}")
print(f"10-Year Average Vacancy (2016-2025): {avg_vac_10yr:.1%}")
print(f"Pre-Shock Stabilized Average (2014-2021): {avg_vac_stabilized:.1%}")

Full History Average Vacancy (26 yrs): 10.8%
10-Year Average Vacancy (2016-2025): 10.6%
Pre-Shock Stabilized Average (2014-2021): 10.1%
